# Day 077 — Exercise 2: frame_to_image and analyze_frame

**What you'll build:** Convert BGR frames to PIL Images and analyze them with a vision LLM.

**Why it matters:** OpenCV stores frames in BGR order; PIL, Ollama, and all Section 5 tools expect RGB. `frame_to_image` bridges the two worlds. `analyze_frame` chains conversion with the Day 76 vision pipeline.

In [ ]:
import numpy as np
from PIL import Image as _PILImage

def _make_mock_frame(h=100, w=100, val=50):
    return np.full((h, w, 3), val, dtype=np.uint8)

class _MockCap:
    def __init__(self, n=5, h=100, w=100):
        self._frames = [_make_mock_frame(h, w) for _ in range(n)]
        self._idx = 0
    def isOpened(self):
        return True
    def read(self):
        if self._idx >= len(self._frames):
            return False, None
        f = self._frames[self._idx]; self._idx += 1
        return True, f
    def release(self):
        pass
    def get(self, prop):
        return 0.0

_mock_camera_fn = lambda device: _MockCap(n=5)
_mock_analyze_fn = lambda img, q: 'FRAME:' + q[:12]


## Task

1. `frame_to_image(frame) -> PIL.Image`
   - `rgb = frame[:, :, ::-1]` — reverse channel axis (BGR → RGB)
   - `return Image.fromarray(rgb)` (from PIL import Image)

2. `analyze_frame(frame, question, analyze_fn=None) -> str`
   - `image = frame_to_image(frame)`
   - If `analyze_fn`: `return analyze_fn(image, question)`
   - Else: BytesIO + PNG save + base64 + `ollama.chat(model='llava', ...)` + return `resp['message']['content']`

## Your Implementation

In [ ]:
import io, base64

def frame_to_image(frame):
    """Convert BGR ndarray to PIL Image in RGB mode."""
    raise NotImplementedError

def analyze_frame(frame, question, analyze_fn=None):
    """Analyze a camera frame with a vision LLM."""
    raise NotImplementedError


In [ ]:
import io, base64

def frame_to_image(frame):
    from PIL import Image
    rgb = frame[:, :, ::-1]
    return Image.fromarray(rgb)

def analyze_frame(frame, question, analyze_fn=None):
    image = frame_to_image(frame)
    if analyze_fn is not None:
        return analyze_fn(image, question)
    import ollama
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': question, 'images': [img_b64]}],
    )
    return resp['message']['content']


## Automated checks

In [ ]:

score, total = 0, 5
try:
    import numpy as np
    from PIL import Image as PILImage

    frame = _make_mock_frame(60, 80)  # (60, 80, 3) BGR
    img = frame_to_image(frame)
    assert isinstance(img, PILImage.Image), f"expected PIL Image, got {type(img)}"
    score += 1; print("✅ frame_to_image returns PIL Image")

    assert img.size == (80, 60), f"size mismatch: {img.size} (expected (80,60))"
    score += 1; print("✅ frame_to_image size correct (width, height)")

    # Check channel reversal: BGR(50,100,150) -> RGB(150,100,50)
    bgr_frame = np.full((10, 10, 3), 0, dtype=np.uint8)
    bgr_frame[:, :, 0] = 50   # B
    bgr_frame[:, :, 1] = 100  # G
    bgr_frame[:, :, 2] = 150  # R
    pil_img = frame_to_image(bgr_frame)
    r, g, b = pil_img.getpixel((0, 0))
    assert r == 150 and g == 100 and b == 50, f"channel flip wrong: R={r},G={g},B={b}"
    score += 1; print("✅ BGR->RGB channel reversal correct")

    result = analyze_frame(frame, 'Q?', analyze_fn=_mock_analyze_fn)
    assert isinstance(result, str)
    score += 1; print("✅ analyze_frame returns str via analyze_fn")

    received = {}
    def _afn(img, q): received.update(img=img, q=q); return 'OK'
    analyze_frame(frame, 'test?', analyze_fn=_afn)
    assert isinstance(received.get('img'), PILImage.Image), "analyze_fn should receive PIL Image"
    assert received.get('q') == 'test?'
    score += 1; print("✅ analyze_fn receives (PIL Image, question)")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
import io, base64

def frame_to_image(frame):
    from PIL import Image
    rgb = frame[:, :, ::-1]
    return Image.fromarray(rgb)

def analyze_frame(frame, question, analyze_fn=None):
    image = frame_to_image(frame)
    if analyze_fn is not None:
        return analyze_fn(image, question)
    import ollama
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': question, 'images': [img_b64]}],
    )
    return resp['message']['content']
```

**Why does `analyze_fn` receive a PIL Image, not the BGR ndarray?** The conversion is an implementation detail. All mock functions and the real Ollama call work with PIL Images, so the mock doesn't need to know or care about BGR arrays.

</details>